# Sentiment scanner — demo notebook

All the logic lives in the `sent_trader` package now; this notebook only drives it and visualizes results.

Setup (once, from the repo root): `pip install -e ".[dev]"`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore

from sent_trader import db, pipeline

## Collect and score

One call scrapes current headlines, downloads the last month of daily bars, stores everything (raw titles, deduplicated), and scores any article the current model hasn't seen yet.

In [ ]:
ticker = 'NVDA'

pipeline.scan(ticker)

## Inspect what's stored

Titles come back raw (no stemming), with the sentiment score and the model version that produced it.

In [ ]:
sentiment = db.get_article_sentiment(ticker)
sentiment.tail(5)

In [ ]:
prices = db.get_stock_prices(ticker)
prices.tail(5)

## Price vs daily average sentiment

VADER compound score interpretation:

| Compound score | Interpreted sentiment |
| --- | --- |
| `>= 0.05` | **Positive** |
| `<= -0.05` | **Negative** |
| between | **Neutral** |

In [ ]:
sentiment_daily = (
    sentiment.groupby(pd.to_datetime(sentiment['publish_date'], utc=True, format='mixed').dt.date)['score']
    .mean()
    .reset_index()
    .rename(columns={'publish_date': 'date', 'score': 'avg_sentiment'})
)

prices['date'] = pd.to_datetime(prices['date'], utc=True, format='mixed').dt.date

merged = pd.merge(prices, sentiment_daily, on='date', how='left')
merged['avg_sentiment'] = merged['avg_sentiment'].fillna(0)

# z-score both series so they are comparable on one axis
merged['price_z'] = zscore(merged['close'])
merged['sent_z'] = zscore(merged['avg_sentiment'])
merged.head(10)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(merged['date'], merged['price_z'], label='Close price (z)')
plt.plot(merged['date'], merged['sent_z'], label='Avg sentiment (z)')
plt.title(f'{ticker}: price vs news sentiment')
plt.xlabel('Date')
plt.legend()
plt.grid()
plt.show()

## Export

Same as `sent-trader export` on the command line.

In [ ]:
stock_df, article_df = db.export_dataframes()
stock_df.to_csv('stock_data.csv', index=False)
article_df.to_csv('article_data.csv', index=False)
len(stock_df), len(article_df)